In [1]:
import typing

"""
    1.Linear 层负责做可学习的线性变换
    2.激活函数负责引入非线性
    3. CrossEntropyLoss 负责将logits转成损失, 并给出反向传播的起点
"""

from typing import override

import dnnlpy.models.mlp as mlp
import numpy as np

rng = np.random.default_rng(42)

print("Numpy Version: ", np.__version__)


Numpy Version:  2.5.2


In [2]:
"""
实现两层的MLP
"""


class MLP(mlp.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.fc1 = mlp.Linear(input_dim, hidden_dim)
        self.relu = mlp.ReLU()
        self.fc2 = mlp.Linear(hidden_dim, num_classes)

    @override
    def forward(self, x: np.ndarray) -> np.ndarray:
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

    @override
    def backward(self, grad: np.ndarray) -> np.ndarray:
        grad = self.fc2.backward(grad)
        grad = self.relu.backward(grad)
        grad = self.fc1.backward(grad)
        return grad


In [3]:
"""
    一次完整的forward 和backward
"""
batch_size = 5
input_dim = 4
hidden_dim = 8
num_classes = 5

x = rng.random((batch_size, input_dim))
y = np.array([0, 1, 2, 1, 0])

model = MLP(input_dim, hidden_dim, num_classes)
loss_fn = mlp.CrossEntropyLoss()

# 执行前向传播
logits = model(x)
loss = loss_fn(logits, y)
print("logits shape:", logits.shape)
print('loss:', loss)

# 执行反向传播
dlogits = loss_fn.backward()
dx = model.backward(dlogits)
for p in model.parameters():
    print(f"Parameter shape:{p.shape}\t |Gradient shape: {p.grad.shape}")



logits shape: (5, 5)
loss: 1.4774774829742665
Parameter shape:(4, 8)	 |Gradient shape: (4, 8)
Parameter shape:(8,)	 |Gradient shape: (8,)
Parameter shape:(8, 5)	 |Gradient shape: (8, 5)
Parameter shape:(5,)	 |Gradient shape: (5,)


In [7]:
"""
参数更新
"""
logits = model(x)
optimizer = mlp.SGD(model.parameters(), lr=0.1)
loss_before = loss_fn(logits, y)

dlogits = loss_fn.backward()
model.backward(dlogits)
optimizer.step()

logits = model(x)
loss_after = loss_fn(logits, y)

print("loss before:", loss_before)
print("loss after:", loss_after)


loss before: 1.4602032984437376
loss after: 1.3781453199131994
